<a href="https://colab.research.google.com/github/gt-cse-6040/bootcamp/blob/main/notebooks/sql/SQL1nb4_SQL_NULLs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SQL Working with NULLs

In [ ]:
!wget https://github.com/gt-cse-6040/bootcamp/raw/main/assets/data/NYC-311-2M_small.db

In [2]:
# create a connection to the database
import sqlite3 as db
import pandas as pd

# Connect to a database (or create one if it doesn't exist)
conn_nyc = db.connect('NYC-311-2M_small.db')

## NULL Values

In many reporting applications, presenting raw NULL values in a final dataset is unwanted or confusing. Also, NULL values can alter the behavior of SQL aggregate functions in ways that lead to inaccurate calculations.

To address this, you will frequently be asked to detect NULL values and replace them with a discrete fallback value (such as converting a blank cell into 0, "N/A", or "Unknown").

SQLite provides two primary functions to handle this value substitution: IFNULL() and COALESCE().

### IFNULL()

The IFNULL() function evaluates a single column or value to determine if it is NULL.

* Syntax: IFNULL(value_to_evaluate, fallback_value)

* If the value is NOT NULL: It returns the original value.

* If the value IS NULL: It intercepts the missing value and returns the fallback_value instead.

See this link for an overview of different sytaxes:  https://www.w3schools.com/sql/sql_isnull.asp

Let's look at a simple example from the NYC 311 Calls database in Notebook 9.

For this exercise we have the subset of data, consisting of one month from 2014.

Recall that not all of the rows have the CITY field filled in.

In [3]:
def ifnull_example():

    # display the list of cities
    query_nulls = '''
                SELECT CITY, COUNT(*)
                FROM DATA
                GROUP BY CITY
                ORDER BY COUNT(*) DESC
                '''
    return query_nulls

df_ifnull_example = pd.read_sql(ifnull_example(),conn_nyc)
display(df_ifnull_example)

,City,COUNT(*)
0,BROOKLYN,45664
1,NEW YORK,31667
2,BRONX,31372
3,None,9244
4,STATEN ISLAND,5222
...,...,...
149,COMMACK,1
150,CINCINATTI,1
151,CHERRYHILL,1
152,BROOKLY,1


We can see that there are over 9,000 rows in which the CITY field is NULL.

When we count(city), that does not count the NULL records

In [4]:
def ifnull_example2():

    # display the list of cities
    query_nulls2 = '''
                SELECT CITY, COUNT(CITY)
                FROM DATA
                GROUP BY CITY
                ORDER BY COUNT(CITY) DESC
                '''
    return query_nulls2

df_ifnull_example2 = pd.read_sql(ifnull_example2(),conn_nyc)
display(df_ifnull_example2)

,City,COUNT(CITY)
0,BROOKLYN,45664
1,NEW YORK,31667
2,BRONX,31372
3,STATEN ISLAND,5222
4,Jamaica,2262
...,...,...
149,CINCINATTI,1
150,CHERRYHILL,1
151,BROOKLY,1
152,BRONXVILLE,1


Rather than leaving blank NULL values in a report—which can look incomplete or confuse end-users—we can replace them with a clear, descriptive label like "No City".

We use the IFNULL() function to handle this substitution.

The syntax is straightforward: SQLite evaluates the CITY field. If it is NULL, the function returns the fallback value "No City"; otherwise, it simply returns the original CITY name.

In [6]:
def ifnull_example3():

    # display the list of cities
    query_nulls3 = '''
                SELECT IFNULL(CITY,"No City") as FULL_CITY, 
                COUNT(IFNULL(CITY,"No City"))
                FROM DATA
                GROUP BY FULL_CITY
                ORDER BY COUNT(FULL_CITY) DESC
                '''
    return query_nulls3

df_ifnull_example3 = pd.read_sql(ifnull_example3(),conn_nyc)
display(df_ifnull_example3)

,FULL_CITY,"COUNT(IFNULL(CITY,""No City""))"
0,BROOKLYN,45664
1,NEW YORK,31667
2,BRONX,31372
3,No City,9244
4,STATEN ISLAND,5222
...,...,...
149,COMMACK,1
150,CINCINATTI,1
151,CHERRYHILL,1
152,BROOKLY,1


### COALESCE() 

In SQLite, when passed exactly two arguments, COALESCE() behaves identically to IFNULL(). Many developers use them interchangeably for basic assignments.

In the next example, you will see COALESCE() evaluate the CITY field and seamlessly substitute "No City" whenever it encounters a NULL value.

However, COALESCE() offers a major functional upgrade over IFNULL() when handling more complex data structures, which we will explore below

In [7]:
def ifnull_example6():

    # display the list of cities
    query_nulls6 = '''
                SELECT COALESCE(CITY,"No City") as FULL_CITY, COUNT(*)
                FROM DATA
                GROUP BY FULL_CITY
                ORDER BY COUNT(FULL_CITY) DESC
                '''
    return query_nulls6

df_ifnull_example6 = pd.read_sql(ifnull_example6(),conn_nyc)
display(df_ifnull_example6)

,FULL_CITY,COUNT(*)
0,BROOKLYN,45664
1,NEW YORK,31667
2,BRONX,31372
3,No City,9244
4,STATEN ISLAND,5222
...,...,...
149,COMMACK,1
150,CINCINATTI,1
151,CHERRYHILL,1
152,BROOKLY,1


We can pass in multiple values to COALESCE, and it will return the FIRST NON-NULL value.

In [8]:
def ifnull_example7():

    # display the list of cities
    query_nulls7 = '''
                SELECT 
                    COALESCE(CITY, Agency, "Unknown") as Location_or_Agency, 
                    COUNT(*) as Total_Calls
                FROM DATA
                GROUP BY Location_or_Agency
                ORDER BY Total_Calls DESC
                '''
    return query_nulls7

df_ifnull_example7 = pd.read_sql(ifnull_example7(),conn_nyc)
display(df_ifnull_example7)

,Location_or_Agency,Total_Calls
0,BROOKLYN,45664
1,NEW YORK,31667
2,BRONX,31372
3,STATEN ISLAND,5222
4,DOT,4873
...,...,...
164,COMMACK,1
165,CINCINATTI,1
166,CHERRYHILL,1
167,BROOKLY,1


### When would we use one versus the other?

Your choice between the two functions comes down to two factors: logic complexity and code portability.

* Use COALESCE() when you need advanced logic or portability. It allows you to build a multi-column fallback chain (an infinite number of substitute columns evaluated left-to-right). Furthermore, because it is the official ANSI SQL standard, it is completely portable—your code will run perfectly if you ever switch from SQLite to PostgreSQL, Oracle, or SQL Server.

* Use IFNULL() for quick, simple substitutions. If you are writing purely in SQLite (or MySQL) and only need to swap a single NULL column for a specific default value, IFNULL() is a perfectly fine shorthand. Just remember that its keyword name changes depending on the database flavor (e.g., SQL Server uses ISNULL(), Oracle uses NVL()).